# Bulk-crystal CXR — analysis & visualization

Loads the checkpoint written by **`cxr_scan.ipynb`** and draws every figure — no
sweep runs here (only the cheap, CPU-only electron transport behind the
penetration figures). The heavy lifting lives in `src/`:

- **`cxr_config.py`** — the shared `Settings` + per-material sweep grids.
- **`cxr_results.py`** — post-processing, the `best_azimuth` reduction, stats table.
- **`cxr_plots.py`** — all the figures (datashader-rasterized trajectories,
  intrinsic spectra, the Eagle XO detector view, parametric heatmaps).

Set `MATERIAL` to match the scan you ran, then run top to bottom.

In [ ]:
import sys

sys.path.insert(0, "src")

# Interactive click-through viewers (browse) work on the inline backend; the
# ipympl widget backend is optional. For a clean PDF export (export_pdf.py),
# switch to `%matplotlib inline` -- browse() then draws every tilt stacked, so
# all figures still render.
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from cxr_config import default_settings, trajectory_sweep, COLLAPSE_AZIMUTH
from cxr_sweep import build_cases
from cxr_results import records, best_azimuth, show_summary, filter_results
from cxr_run import load_checkpoint, cases_from_results
from cxr_plots import (
    browse,
    plot_heatmaps,
    plot_eaglexo_efficiency,
    plot_eaglexo_measured,
    plot_timepix_efficiency,
    plot_timepix_poisson,
    plot_trajectory_grid,
    plot_penetration_profile,
)

In [ ]:
# Material must match the scan you ran in cxr_scan.ipynb.
MATERIAL = "ptse2"

settings = default_settings()
results = load_checkpoint(MATERIAL)          # {name: {E0: record}}
cases = cases_from_results(results)          # rebuild the case list from the records
res = filter_results(results, cases)         # all loaded cases for this material

# Photon-counting stats (best azimuth per polar tilt + energy).
recs = best_azimuth(records(res)) if COLLAPSE_AZIMUTH else records(res)
show_summary(recs, settings)

## Intrinsic spectra

One figure per polar tilt, click through with the tilt slider. `chunk` is the
best-azimuth total / CXR-only pair; `by_energy` overlays every beam energy;
`full` is the full measured range (sharp lines on the wide brem, log-log).

In [ ]:
browse(res, settings, kind="chunk")        # best-azimuth: total | CXR-only

In [ ]:
browse(res, settings, kind="by_energy")    # every beam energy overlaid

In [ ]:
browse(res, settings, kind="full")         # full measured range, log-log

In [ ]:
# Parametric maps over (polar x azimuth), one panel per beam energy, one figure
# per quantity (peak flux, line energy, FWHM, line/total, line flux, total flux).
plot_heatmaps(res, settings, cases=cases, line_metric="prominence");

## Eagle XO detector response

The Raptor **Eagle XO** direct-detection CCD modelled as `solid_angle x QE(E)`
(`eaglexo_response.py`): soft PXR lines pass at ~90% QE while the hard brem is
crushed by the thin back-thinned sensor. This is the detector view (it replaces
the old EDS detector-convolved spectra).

> Point the sweep at the real Eagle solid angle with
> `material_sweep(..., **eaglexo_response.sweep_geometry("4240", distance_m=...))`,
> and set the working distance in `eaglexo_response.py` before trusting absolute rates.

In [ ]:
plot_eaglexo_efficiency(sensor="4240");      # QE + solid angle + resolution

In [ ]:
browse(res, settings, kind="eaglexo")       # detected (solid) vs incident (dotted)

In [ ]:
plot_eaglexo_measured(res, settings, integration_s=600.0);  # Poisson "measured"

## Timepix3 detector response (optional comparison)

The 2×2 Timepix3 quad forward model (`timepix_response.py`): photoabsorption →
charge sharing → per-pixel threshold counting → ToT noise → Poisson. Set the real
quad thickness / bias — σ_diff ∝ 1/√bias is the most sensitive knob.

In [ ]:
TPX_THICKNESS_UM = 300.0  ### FILL IN -- Si sensor thickness [um]
TPX_BIAS_V = 100.0        ### FILL IN -- applied bias [V]

plot_timepix_efficiency(thickness_um=TPX_THICKNESS_UM, bias_v=TPX_BIAS_V)
plot_timepix_poisson(
    res, settings, integration_s=600.0,
    thickness_um=TPX_THICKNESS_UM, bias_v=TPX_BIAS_V,
)
browse(
    res, settings, kind="timepix",
    thickness_um=TPX_THICKNESS_UM, bias_v=TPX_BIAS_V,
)

## Electron penetration

Cross-sections of the electron cascade in the beam-detector plane,
datashader-rasterized and coloured by electron energy along each track. All
panels at one beam energy share ONE frame, so across tilts only the slab rotates
(red = beam, green = detector). Then the mean-energy and electron-lifetime
profiles vs penetration depth. These run the cheap, CPU-only electron transport
directly — no checkpoint required.

In [ ]:
traj_sweep = trajectory_sweep(MATERIAL, n_tilts=9, energies=(30, 60))
traj_cases = build_cases(traj_sweep, settings.n_electrons, settings.n_electrons_brem)

# one penetration grid (polar-tilt panels) per beam energy
for E0 in sorted({c["E0_keV"] for c in traj_cases}):
    plot_trajectory_grid(traj_cases, energy=E0, Ne=120)

# mean electron energy + lifetime (age) vs penetration depth
plot_penetration_profile(traj_cases, Ne=500)
plt.show()